## Gold — OD flows

Calculates consecutives origin-destination stops pairs in the same trip.

### Source
- `gtfs_silver.stop` — stop list with stop names
- `gtfs_silver.trips` — trips linked to routes and services
- `gtfs_silver.stop_times` — stops served per trip

### Output
`gtfs_gold.od_flows`

| Column | Description |
|--------|-------------|
| city | City name |
| origin_stop_id | origin id |
| destination_stop_id | destination id |
| route_id | route id |
| n_trips | number of trips from origin to destination |
| avg_travel_time_minutes | average travel time in minutes |
| origin_stop_name | origin name |
| destination_stop_name | destination name |

### CTE Logic
- **consecutive_pairs** — joins `stop_times` → `stop_times` to find the pairs of consecutives origin and destination stops


Logica: self-join di stop_times dove la seconda fermata ha stop_sequence = prima + 1 sullo stesso trip_id e city.

In [0]:
CREATE OR REPLACE TABLE gtfs_gold.od_flows AS
WITH consecutive_pairs AS (
    SELECT 
        st1.trip_id,
        st1.city,
        st1.stop_id AS origin_stop_id,
        st2.stop_id AS destination_stop_id,
        st1.departure_secs,
        st2.arrival_secs,
        (st2.arrival_secs - st1.departure_secs)/60 AS travel_time_minutes
    FROM gtfs_silver.stop_times st1
    JOIN gtfs_silver.stop_times st2 ON st1.stop_sequence = st2.stop_sequence - 1 AND st1.trip_id = st2.trip_id  AND st1.city=st2.city
    WHERE st2.arrival_secs > st1.departure_secs AND (st2.arrival_secs - st1.departure_secs) < 3600
)
SELECT
    t.route_id,
    t.city,
    cp.origin_stop_id,
    cp.destination_stop_id,
    ROUND(AVG(cp.travel_time_minutes),2) as avg_travel_time_minutes,
    COUNT(DISTINCT t.trip_id) as num_trips,
    s.stop_name AS origin_stop_name,
    s2.stop_name AS destination_stop_name
FROM gtfs_silver.trips t 
JOIN consecutive_pairs cp ON t.trip_id = cp.trip_id AND t.city=cp.city
JOIN gtfs_silver.stops s ON cp.origin_stop_id = s.stop_id AND s.city=cp.city
JOIN gtfs_silver.stops s2 ON cp.destination_stop_id = s2.stop_id AND s2.city=cp.city
GROUP BY t.route_id, t.city, cp.origin_stop_id, cp.destination_stop_id, origin_stop_name, destination_stop_name;


In [0]:
SELECT city, origin_stop_name, destination_stop_name, num_trips, avg_travel_time_minutes
FROM gtfs_gold.od_flows
ORDER BY num_trips DESC
LIMIT 10